#### Import Llava

In [22]:
!pip install grakel
!pip install networkx
!pip install spacy
!python -m spacy download en_core_web_sm

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached GraKeL-0.1.10-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (11 kB)
  Using cached Cython-3.0.10-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.2 kB)
  Using cached future-1.0.0-py3-none-any.whl.metadata (4.0 kB)
Using cached GraKeL-0.1.10-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (1.9 MB)
Using cached Cython-3.0.10-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.6 MB)
Using cached future-1.0.0-py3-none-any.whl (491 kB)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached spacy-3.7.5-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (27 kB)
  Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl.metadata (2.8 kB)
  Using cached spacy_loggers-1.0.5-py3-none-any.whl.metadata (23 kB)
  Using cached murmurhash-1.0.10-cp310-cp310-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.0 kB)
  Using cached cymem-2.0.8-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (8.4 kB)
  Using cached preshed-3.0.9-cp310-cp310-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.2 kB)
  Using cached wasabi-1.1.3-py3-none-any.whl.metadata (28 kB)
  Using cached srsly-2.4.8-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (20 kB)
  Using cached catalogue-2.0.10-py3-none-any.whl.metadata (14 kB)
  Using cached weasel-0.4.1-py3-none-any.whl.metadata (4.6 kB)
  Using cached langcodes-3.4.0-py3-none-any.whl.metadata (29 k

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 78.9 MB/s eta 0:00:0000:010:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [5]:
import warnings
warnings.filterwarnings("ignore")

import argparse
import torch
from llava.constants import (
    IMAGE_TOKEN_INDEX,
    DEFAULT_IMAGE_TOKEN,
    DEFAULT_IM_START_TOKEN,
    DEFAULT_IM_END_TOKEN,
    IMAGE_PLACEHOLDER,
)
from llava.conversation import conv_templates, SeparatorStyle
from llava.model.builder import load_pretrained_model
from llava.utils import disable_torch_init
from llava.mm_utils import (
    process_images,
    tokenizer_image_token,
    get_model_name_from_path,
)

from PIL import Image
import requests
from io import BytesIO
import re
import time

# Decorator to measure inference time
def timeit_decorator(func):
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        end_time = time.perf_counter()
        print(f"Inference time: {end_time - start_time:.8f} seconds")
        return result
    return wrapper

# Function to parse image files
def image_parser(args):
    out = args.image_file.split(args.sep)
    return out

# Function to load an image
def load_image(image_file):
    if image_file.startswith("http") or image_file.startswith("https"):
        response = requests.get(image_file)
        image = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        image = Image.open(image_file).convert("RGB")
    return image

# Function to load multiple images
def load_images(image_files):
    out = []
    for image_file in image_files:
        image = load_image(image_file)
        out.append(image)
    return out

# Function to load the model
def load_model(args):
    disable_torch_init()
    model_name = get_model_name_from_path(args.model_path)
    tokenizer, model, image_processor, context_len = load_pretrained_model(
        args.model_path, args.model_base, model_name
    )
    return model_name, tokenizer, model, image_processor, context_len

# Function to evaluate the model
@timeit_decorator
def eval_model(args, model_name, tokenizer, model, image_processor):
    # Prepare the query
    qs = args.query
    image_token_se = DEFAULT_IM_START_TOKEN + DEFAULT_IMAGE_TOKEN + DEFAULT_IM_END_TOKEN
    if IMAGE_PLACEHOLDER in qs:
        if model.config.mm_use_im_start_end:
            qs = re.sub(IMAGE_PLACEHOLDER, image_token_se, qs)
        else:
            qs = re.sub(IMAGE_PLACEHOLDER, DEFAULT_IMAGE_TOKEN, qs)
    else:
        if model.config.mm_use_im_start_end:
            qs = image_token_se + "\n" + qs
        else:
            qs = DEFAULT_IMAGE_TOKEN + "\n" + qs

    # Determine conversation mode
    if "llama-2" in model_name.lower():
        conv_mode = "llava_llama_2"
    elif "mistral" in model_name.lower():
        conv_mode = "mistral_instruct"
    elif "v1.6-34b" in model_name.lower():
        conv_mode = "chatml_direct"
    elif "v1" in model_name.lower():
        conv_mode = "llava_v1"
    elif "mpt" in model_name.lower():
        conv_mode = "mpt"
    else:
        conv_mode = "llava_v0"

    if args.conv_mode is not None and conv_mode != args.conv_mode:
        print(
            "[WARNING] the auto inferred conversation mode is {}, while `--conv-mode` is {}, using {}".format(
                conv_mode, args.conv_mode, args.conv_mode
            )
        )
    else:
        args.conv_mode = conv_mode

    # Construct conversation
    conv = conv_templates[args.conv_mode].copy()
    conv.append_message(conv.roles[0], qs)
    conv.append_message(conv.roles[1], None)
    prompt = conv.get_prompt()

    # Load images
    image_files = image_parser(args)
    images = load_images(image_files)
    image_sizes = [x.size for x in images]
    images_tensor = process_images(
        images,
        image_processor,
        model.config
    ).to(model.device, dtype=torch.float16)

    # Tokenize input
    input_ids = (
        tokenizer_image_token(prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt")
        .unsqueeze(0)
        .cuda()
    )

    # Generate output
    with torch.inference_mode():
        output_ids = model.generate(
            input_ids,
            images=images_tensor,
            image_sizes=image_sizes,
            do_sample=True if args.temperature > 0 else False,
            temperature=args.temperature,
            top_p=args.top_p,
            num_beams=args.num_beams,
            max_new_tokens=args.max_new_tokens,
            use_cache=True,
        )

    # Decode output
    outputs = tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0].strip()
    return outputs

# Example usage in a notebook
class Args:
    model_path = "liuhaotian/llava-v1.6-mistral-7b"
    model_base = None
    image_file = "/home/ubuntu/Multimodal-Uncertainty-Quantification/playground/LLaVA/images/demo_cli.gif"  # or URL
    query = "Describe this image."
    conv_mode = None
    sep = ","
    temperature = 0.2
    top_p = None
    num_beams = 1
    max_new_tokens = 512

args = Args()

# Load model once
model_name, tokenizer, model, image_processor, context_len = load_model(args)
eval_model(args, model_name, tokenizer, model, image_processor)
# Get responses
# for _ in range(20):
#     response = eval_model(args, tokenizer, model, image_processor)
#     print(response)


Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.15s/it]
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Inference time: 12.62430946 seconds


'The image displays a computer screen with a terminal window open. The terminal window shows a command prompt with a user input. The command prompt reads:\n\n```\n[223-07-29 18:32:19.966] INF: Real accelerator_py:110:Get accelerator_py:110:Setting ds_acc\n```\n\nBelow the command prompt, there is a text box with a file path:\n\n```\n/home/user/checkpoint_shards: 108%\n```\n\nThe text box is empty, indicating that the user has not yet entered a command or file path. The background of the terminal window is a dark gray color, which is typical for many terminal emulators. The overall style of the image is a simple, utilitarian representation of a command-line interface.'

#### Load Examples from MMT-Bench and UQ

In [8]:
import pandas as pd
import base64
from PIL import Image
from io import BytesIO


# Read the TSV file into a DataFrame
df = pd.read_csv('/home/ubuntu/Multimodal-Uncertainty-Quantification/datasets/MMT-Bench/MMT-Bench_VAL.tsv', sep='\t')
filtered_df_hallucination = df[df['category'].str.contains('hallucination')]
filtered_df_hallucination.shape

(80, 16)

In [9]:
filtered_df_hallucination.head()

with pd.option_context('display.max_colwidth', None):
    question = filtered_df_hallucination.loc[2787, 'question']
    choice_a = filtered_df_hallucination.loc[2787, 'A']
    choice_b = filtered_df_hallucination.loc[2787, 'B']
    choice_c = filtered_df_hallucination.loc[2787, 'C']
    choice_d = filtered_df_hallucination.loc[2787, 'D']

    prompt = f"{question}\nA. {choice_a}\nB. {choice_b}\nC. {choice_c}\nD. {choice_d}"
    print(prompt)

Question: Based on the given image, which caption has the most correct ordering of the constituents?

Choose the best answer from the following choices.
A. Four people are playing soccer on a beach.
B. Four people are playing football by a body of water in partly cloudy weather.
C. A group of people playing on a beach.
D. Some people are kicking a white ball on a beach.


In [10]:
### Save the byted coded image to png to load
image_id = 2787
image_data_base64 = filtered_df_hallucination.loc[image_id, 'image']
image_data = base64.b64decode(image_data_base64)
image = Image.open(BytesIO(image_data))
image.save('decoded_image_{}.png'.format(image_id))

# Give the prompt and the image to Llava
Args.image_file = 'decoded_image_{}.png'.format(image_id)
Args.query = prompt

# Get the response
# args = Args()
# eval_model(args)

In [19]:
import spacy
from grakel import GraphKernel, Graph
import networkx as nx
import numpy as np
import random
import base64
from PIL import Image
from io import BytesIO

# Load SpaCy model
nlp = spacy.load("en_core_web_sm")

# Function to decode and save image
def decode_and_save_image(image_data_base64, image_id):
    image_data = base64.b64decode(image_data_base64)
    image = Image.open(BytesIO(image_data))
    image_file = f'decoded_image_{image_id}.png'
    image.save(image_file)
    return image_file

# Function to extract entities and relationships using SpaCy
def extract_entities_and_relationships(text):
    doc = nlp(text)
    entities = [(ent.text, ent.label_) for ent in doc.ents]
    relationships = []
    for token in doc:
        if token.dep_ == "ROOT":
            for child in token.children:
                if child.dep_ in ["nsubj", "dobj", "prep"]:
                    relationships.append((token.text, child.text))
    if not entities:
        for chunk in doc.noun_chunks:
            entities.append((chunk.text, 'NOUN_CHUNK'))
    return entities, relationships

# Function to construct graph from entities and relationships
def construct_graph(entities, relationships):
    G = nx.DiGraph()
    for entity, label in entities:
        G.add_node(entity, label=label)
    for subj, obj in relationships:
        if not G.has_node(subj):
            G.add_node(subj, label="unknown")
        if not G.has_node(obj):
            G.add_node(obj, label="unknown")
        G.add_edge(subj, obj)
    return G

# Function to convert NetworkX graphs to GraKeL graphs
def convert_to_grakel_graphs(graphs):
    grakel_graphs = []
    for G in graphs:
        if len(G.nodes) > 0 and len(G.edges) > 0:
            # node_labels = {i: G.nodes[node]['label'] for i, node in enumerate(G.nodes)}
            # edges = [(list(G.nodes).index(u), list(G.nodes).index(v)) for u, v in G.edges()]
            # adjacency = nx.adjacency_matrix(G).todense().tolist()
            # grakel_graphs.append(Graph(adjacency=adjacency, node_labels=node_labels))
            node_labels = {i: G.nodes[node]['label'] for i, node in enumerate(G.nodes)}
            edges = [(list(G.nodes).index(u), list(G.nodes).index(v)) for u, v in G.edges()]
            grakel_graphs.append(Graph(edges, node_labels=node_labels))

    return grakel_graphs

# Function to compute Weisfeiler-Lehman kernel and calculate uncertainty
def calculate_uncertainty(graphs):
    grakel_graphs = convert_to_grakel_graphs(graphs)
    
    if not grakel_graphs:
        print("No valid graphs were generated.")
        return None

    gk = GraphKernel(kernel={"name": "weisfeiler_lehman"}, normalize=True)
    
    try:
        K = gk.fit_transform(grakel_graphs)
        pairwise_distances = 1 - K
        uncertainty = np.mean(pairwise_distances)
    except Exception as e:
        print(f"Error during kernel computation: {e}")
        uncertainty = None
    
    return uncertainty

# filtered_df_hallucination = pd.read_csv("path_to_your_filtered_df_hallucination.csv")  # Load your DataFrame

# Assuming `eval_model` and `Args` are already defined as in previous examples
def generate_responses_and_calculate_uncertainty(filtered_df_hallucination, num_images=5):
    random_indices = random.sample(filtered_df_hallucination.index.tolist(), num_images)
    results = {}
    
    for idx in random_indices:
        with pd.option_context('display.max_colwidth', None):
            question = filtered_df_hallucination.loc[idx, 'question']
            choice_a = filtered_df_hallucination.loc[idx, 'A']
            choice_b = filtered_df_hallucination.loc[idx, 'B']
            choice_c = filtered_df_hallucination.loc[idx, 'C']
            choice_d = filtered_df_hallucination.loc[idx, 'D']

            prompt = f"{question}\nA. {choice_a}\nB. {choice_b}\nC. {choice_c}\nD. {choice_d}"
        
        # Decode and save the image
        image_data_base64 = filtered_df_hallucination.loc[idx, 'image']
        image_file = decode_and_save_image(image_data_base64, idx)

        # Set the arguments
        Args.image_file = image_file
        Args.query = prompt
        args = Args()

        # Load model once
        model_name, tokenizer, model, image_processor, context_len = load_model(args)

        responses = []
        for _ in range(4):  # Get 20 responses
            response = eval_model(args, model_name, tokenizer, model, image_processor)
            responses.append(response)
        
        # Extract entities and relationships from responses
        graphs = []
        for response in responses:
            entities, relationships = extract_entities_and_relationships(response)
            print('Entities', entities)
            print('Relationships', relationships)
            G = construct_graph(entities, relationships)
            graphs.append(G)
        
        # Calculate uncertainty
        uncertainty = calculate_uncertainty(graphs)
        
        if uncertainty is None:
            print(f"Uncertainty could not be calculated for index {idx}.")
        
        results[idx] = {
            'prompt': prompt,
            'responses': responses,
            'uncertainty': uncertainty
        }
    
    return results

# Usage example:
# results = generate_responses_and_calculate_uncertainty(filtered_df_hallucination, num_images=5)

# Print results
# for idx, result in results.items():
#     print(f"Image ID: {idx}")
#     print(f"Prompt: {result['prompt']}")
#     print(f"Uncertainty: {result['uncertainty']:.4f}")
#     print("Sample Responses:", result['responses'][:3])  # Print first 3 responses as a sample
#     print()


In [20]:
# Usage example:
results = generate_responses_and_calculate_uncertainty(filtered_df_hallucination, num_images=5)

# Print results
for idx, result in results.items():
    print(f"Image ID: {idx}")
    print(f"Prompt: {result['prompt']}")
    print(f"Uncertainty: {result['uncertainty']:.4f}")
    print("Sample Responses:", result['responses'][:3])  # Print first 3 responses as a sample
    print()

Loading checkpoint shards: 100%|██████████| 4/4 [00:06<00:00,  1.50s/it]
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Inference time: 1.62172773 seconds


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Inference time: 1.61627014 seconds


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Inference time: 5.63221793 seconds


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Inference time: 6.16601143 seconds
Inference time: 5.80309315 seconds
Entities [('A. Middle -', 'PERSON'), ('B. Bottom', 'PERSON'), ('C. Top -', 'PERSON'), ('D. nan -', 'PRODUCT')]
Relationships [('shows', 'image'), ('shows', 'part'), ('show', 'image'), ('show', 'part'), ('show', 'image'), ('show', 'part'), ('contain', 'image'), ('contain', 'nanotechnology')]
Entities [('A. Middle -', 'PERSON'), ('B. Bottom', 'PERSON'), ('C. Top -', 'PERSON'), ('D. nan -', 'PRODUCT')]
Relationships [('shows', 'image'), ('shows', 'part'), ('shows', 'from'), ('show', 'image'), ('show', 'part'), ('shows', 'image'), ('shows', 'part'), ('shows', 'from'), ('contain', 'image'), ('contain', 'nanotechnology')]
Entities [('A. Middle -', 'PERSON'), ('B. Bottom', 'PERSON'), ('C. Top -', 'PERSON'), ('D. nan -', 'PRODUCT')]
Relationships [('shows', 'image'), ('shows', 'section'), ('show', 'image'), ('show', 'section'), ('show', 'image'), ('show', 'section'), ('is', 'This')]
Entities [('A. Middle -', 'PERSON'), ('B. 

TypeError: unsupported format string passed to NoneType.__format__

In [22]:
results

{2820: {'prompt': 'Generate attributes related to objects in a natural image\nA. Skirt\nB. Shorts\nC. Pants\nD. nan',
  'responses': ['A. Skirt\nB. Shorts\nC. Pants\nD. None of the above',
   'A. Skirt\nB. Shorts\nC. Pants\nD. None of the above',
   'In the image, the woman is wearing a white crop top and high-waisted shorts with a floral pattern. She is also holding a handbag. The background shows a city street with buildings, a sidewalk, and some holiday lights. The woman is wearing sunglasses and has her hair styled in loose waves.',
   'A. Skirt'],
  'uncertainty': 0.0},
 2799: {'prompt': 'Question: Based on the given image, which caption has the most correct ordering of the constituents?\n\nChoose the best answer from the following choices.\nA. An older man sites on a cushioned bench with his hands crossed in his lap.\nB. An elderly man sleeps sitting up on the end of a red couch\nC. A man rests his eyes, sitting on a red couch with his hands folded.\nD. Old man wearing a hat and 

#### Load Examples from MM-Bench and UQ